# FCA–MoE v0.3 — Real GPT-OSS-20B routing extraction

**Model:** `openai/gpt-oss-20b` · **Architecture:** 24 MoE layers, 32 experts, Top-4 · **Dataset:** WikiText-2 raw test split.

This notebook captures the original pretrained model's actual router outputs through forward hooks on each `GptOssTopKRouter`. It records selected expert IDs, selected routing weights, and selected router logits. It does not alter the router or generate synthetic empirical observations.

## Critical Colab hardware gate

GPT-OSS-20B ships MXFP4 expert weights. The upstream Transformers deployment guidance specifies Hopper-class SM90+ for the optimized MXFP4 path. A Colab T4 (SM75) is **not compatible**. CPU offloading can reduce memory pressure but cannot make an unsupported MXFP4 GPU kernel compatible. The extraction script therefore stops before model download on a T4.

Use a compatible SM90+ GPU runtime with sufficient host RAM. If the available Colab runtime is T4-only, do not claim real GPT-OSS traces were collected; obtain access to a supported GPU or approve a different model/quantization strategy separately.

Reference: [OpenAI gpt-oss model card](https://deploymentsafety.openai.com/gpt-oss), [OpenAI Transformers deployment guide](https://developers.openai.com/cookbook/articles/gpt-oss/run-transformers).

In [ ]:
# Install runtime dependencies; retain Colab's preinstalled PyTorch.
%pip -q install -U "transformers>=4.55.0" accelerate datasets pyarrow pandas psutil "triton==3.4.0" kernels

In [ ]:
# Inspect hardware before downloading any model weights.
import torch, psutil, platform
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Available host RAM GiB:", round(psutil.virtual_memory().available / 2**30, 2))
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    free, total = torch.cuda.mem_get_info(0)
    print("GPU:", name, "compute capability:", cap)
    print("GPU free/total GiB:", round(free/2**30,2), "/", round(total/2**30,2))
    if cap < (9, 0):
        print("STOP: this GPU is not supported for the upstream GPT-OSS MXFP4 kernel path. T4 is SM75; CPU offload does not resolve this.")
else:
    print("STOP: CUDA GPU not detected.")

## Mount persistent storage and obtain the approved research branch

The Parquet shards and manifest are written to Google Drive. The temporary model offload directory remains on the Colab VM's local disk for better random-access performance.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
REPO_DIR = pathlib.Path('/content/FCA-MoE')
if not REPO_DIR.exists():
    !git clone --depth 1 --branch research/fca-moe-v0.3 https://github.com/msellamiTN/FCA-MoE.git /content/FCA-MoE
%cd /content/FCA-MoE/research/v03


In [ ]:
# Run a small real-model pilot: four 256-token sequences.
# This command intentionally fails fast on T4/SM75 before downloading model weights.
OUTPUT_DIR = "/content/drive/MyDrive/FCA_MoE_v03/gpt_oss_20b_wikitext_test"
!python scripts/extract_gpt_oss_routes.py \
  --output-dir "$OUTPUT_DIR" \
  --sequence-length 256 \
  --max-chunks 4 \
  --gpu-memory 32GiB \
  --cpu-memory 20GiB \
  --min-cpu-ram-gib 24


In [ ]:
# Validate the pilot shards after extraction completes.
!python scripts/validate_gpt_oss_traces.py --output-dir "$OUTPUT_DIR"

## Resume after a Colab disconnect

Rerun the extraction command with the exact same output directory and configuration. The script scans contiguous committed Parquet shards and resumes from the next missing sequence chunk. Do not change sequence length, dataset, model revision, or architecture for an existing checkpoint directory.

To increase the pilot from 4 to 100 chunks, change only `--max-chunks 100`. Use `--max-chunks 0` to stream the full dataset split. The manifest records the model revision, data source, architecture, phase and completed shards.